# Generating RAG Answers for Evaluation

`02-search-eval.ipynb` scored *retrieval* — whether the right document comes back. But the thing users actually see is the LLM's generated answer, and a good retrieval doesn't guarantee a good answer. This notebook runs the full RAG pipeline (search + prompt + LLM) over all 395 ground-truth questions and saves `(question, answer_llm, answer_orig, document)` records. The next notebook, `04-llm-judge.ipynb`, uses these records to score answer quality with an LLM judge.

## 1. Loading Ground Truth and Rebuilding the Index

Same ground truth and `llm-zoomcamp` index as `02-search-eval.ipynb`. `doc_idx` is a `document_id -> document` lookup, so the *original* FAQ answer for each ground-truth question can be pulled up alongside whatever the RAG pipeline generates.

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
ground_truth[10]

In [ ]:
from zoomcamp.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [ ]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [ ]:
q = ground_truth[10]
q

In [ ]:
doc_idx[q['document']]

## 2. `RAGWithUsage`: a RAG Assistant That Tracks Cost

`RAGWithUsage` (in `zoomcamp/evaluation_utils.py`) subclasses `RAGBase` from module 01, keeping the same `search → build_prompt → llm → rag` pipeline but recording every LLM call's `usage` into `self.usages` as it goes. `total_cost()` sums them with `calc_total_price`, and `reset_usage()` clears the log — useful for measuring the cost of just one batch of calls instead of the running total.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from zoomcamp.evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [ ]:
q['question']

In [ ]:
answer = assistant.rag(q['question'])

In [ ]:
assistant.total_cost()

In [ ]:
print(answer)

In [ ]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

In [ ]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

## 3. Answering a Single Ground-Truth Question

`assistant.rag(q['question'])` runs the whole pipeline for one question (cost: ≈$0.0019 for this call). Pairing the generated `answer_llm` with the FAQ's own `answer_orig` for the same `document` id gives one evaluation record — the LLM answer and the ground truth it should match, side by side.

In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [ ]:
record = generate_rag_answer(q)
record

In [ ]:
assistant.total_cost()

In [ ]:
assistant.reset_usage()

In [ ]:
assistant.total_cost()

## 4. Wrapping It in `generate_rag_answer`

`generate_rag_answer(rec)` packages Section 3's steps into a function: look up the original document, run `assistant.rag`, and return the full record. `assistant.reset_usage()` zeroes the running cost counter right before the full-dataset run, so `total_cost()` afterward reflects only that run's spend.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from zoomcamp.evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

In [ ]:
results[:10]

In [ ]:
df_results = pd.DataFrame(results)

In [ ]:
df_results.head()

## 5. Generating Answers for All 395 Questions

Same pattern as `01-data-gen.ipynb`: `map_progress` fans `generate_rag_answer` out across a 6-worker `ThreadPoolExecutor`, running RAG end-to-end (search + LLM call) for every ground-truth question in parallel instead of one at a time.

In [ ]:
assistant.total_cost()


## 6. Total Cost and Saving the Dataset

395 full RAG round trips (search + LLM generation) cost **≈$0.343** in total. The resulting `(question, answer_llm, answer_orig, document)` records are saved to `data/rag-answers-new.csv`, which `04-llm-judge.ipynb` loads next to score how well `answer_llm` matches `answer_orig`.

In [ ]:
df_results.to_csv("./data/rag-answers.csv", index=False)